In [1]:
import os
import random
from pathlib import Path
import json
import math
import time
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from PIL import Image

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, datasets, models
from torchvision.utils import make_grid
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import StratifiedShuffleSplit
from scipy.stats import ttest_ind
import matplotlib.pyplot as plt

# Configurations

In [3]:
DATA_ROOT = os.getenv('DATA_ROOT', '/kaggle/input/cucumber-dataset/Original Image')
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
SPLIT_DIR = os.path.join(OUTPUT_DIR, 'splits')
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')
XAI_DIR = os.path.join(OUTPUT_DIR, 'xai')
CKPT_DIR = os.path.join(OUTPUT_DIR, 'checkpoints')
for p in [SPLIT_DIR, PLOTS_DIR, XAI_DIR, CKPT_DIR]: os.makedirs(p, exist_ok=True)

RNG_SEED = 42
torch.manual_seed(RNG_SEED); np.random.seed(RNG_SEED); random.seed(RNG_SEED)

# 8 classes, 160 images each, 80/10/10 => train: 1024, val: 128, test: 128
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

# episode settings (classic prototypical few-shot)
N_WAY = 8
K_SHOT = 5   # support per class
Q_QUERY = 15 # query per class
EPISODES_PER_EPOCH = 10       # reduced from 25 for speed (was bottleneck)
VAL_EPISODES = 5              # reduced from 10 for speed
TEST_EPISODES = 10            # reduced from 25 for speed

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# GPU optimization
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
    print(f'  CUDA Version: {torch.version.cuda}')
    print(f'  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')
else:
    print('⚠ GPU NOT detected. Training on CPU (will be slow).')

✓ GPU available: Tesla P100-PCIE-16GB
  CUDA Version: 12.4
  GPU Memory: 17.1GB


# Utilities: file, split, dataset

In [5]:
def make_stratified_splits(root_dir, output_dir=SPLIT_DIR, seed=RNG_SEED):
    """Split the dataset in a stratified manner and save train/val/test CSV files."""
    data = []
    root = Path(root_dir)
    classes = sorted([d.name for d in root.iterdir() if d.is_dir()])
    if len(classes) < 2:
        raise ValueError('Dataset root must contain class subfolders')

    for lbl, cls in enumerate(classes):
        images = list((root / cls).glob('*'))
        images = [x for x in images if x.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']]
        for img in images:
            data.append({'image': str(img), 'label': lbl, 'class': cls})

    df = pd.DataFrame(data)
    x = df['image']; y = df['label']

    # split train / temp (val+test)
    splitter1 = StratifiedShuffleSplit(n_splits=1, test_size=(1-TRAIN_RATIO), random_state=seed)
    train_idx, temp_idx = next(splitter1.split(x, y))

    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_temp = df.iloc[temp_idx].reset_index(drop=True)

    splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=TEST_RATIO/(VAL_RATIO+TEST_RATIO), random_state=seed)
    val_idx, test_idx = next(splitter2.split(df_temp['image'], df_temp['label']))

    df_val = df_temp.iloc[val_idx].reset_index(drop=True)
    df_test = df_temp.iloc[test_idx].reset_index(drop=True)

    df_train.to_csv(os.path.join(output_dir, 'train.csv'), index=False)
    df_val.to_csv(os.path.join(output_dir, 'val.csv'), index=False)
    df_test.to_csv(os.path.join(output_dir, 'test.csv'), index=False)

    print('Split sizes: train={}, val={}, test={}'.format(len(df_train), len(df_val), len(df_test)))
    return df_train, df_val, df_test

In [6]:
class ImagePathsDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image']).convert('RGB')
        image = transforms.ToTensor()(image)

        if self.transform is not None:
            image = self.transform(image)

        return image, int(row['label'])

In [7]:
class EpisodicSampler:
    def __init__(self, labels, n_way, k_shot, q_query, episodes, seed=RNG_SEED):
        self.labels = np.array(labels)
        self.n_way = n_way
        self.k_shot = k_shot
        self.q_query = q_query
        self.episodes = episodes
        self.rng = np.random.RandomState(seed)

        self.by_class = {c: np.where(self.labels == c)[0] for c in np.unique(self.labels)}
        # Check if each class has enough samples
        for c, idx in self.by_class.items():
            if len(idx) < self.k_shot + self.q_query:
                raise ValueError(f'Class {c} has {len(idx)} samples, but needs at least {self.k_shot + self.q_query} for k_shot={self.k_shot}, q_query={self.q_query}')

    def __len__(self):
        return self.episodes

    def __iter__(self):
        for _ in range(self.episodes):
            selected_classes = self.rng.choice(list(self.by_class.keys()), size=self.n_way, replace=False)
            support_idx = []
            query_idx = []
            for c in selected_classes:
                choices = self.rng.choice(self.by_class[c], size=self.k_shot + self.q_query, replace=False)
                support_idx.extend(choices[:self.k_shot].tolist())
                query_idx.extend(choices[self.k_shot:].tolist())
            yield support_idx, query_idx

In [8]:
def make_transforms():
    train_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    eval_transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_transform, eval_transform

# Model: Prototypical network embedding + prototype compute

In [9]:
class ConvEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        x = self.encoder(x)    # [B, 512, 1, 1]
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

In [10]:
class ProtoNet(nn.Module):
    def __init__(self, embedding_net):
        super().__init__()
        self.embedding_net = embedding_net

    def forward(self, support, support_labels, query):
        # support: [N*K, C, H, W], query: [N*Q, C, H, W]
        z_support = self.embedding_net(support)   # [N*K, D]
        z_query = self.embedding_net(query)       # [N*Q, D]

        # compute prototypes per class: mean of support embeddings
        unique_labels = torch.unique(support_labels)
        # p_c = (1/|S_c|) * sum_{x in S_c} f(x)
        prototypes = []
        for c in unique_labels:
            class_embeddings = z_support[support_labels == c]
            prototypes.append(class_embeddings.mean(dim=0))
        prototypes = torch.stack(prototypes)

        # compute squared L2 distance to prototypes
        # Note: pairwise distance matrix is [Q, N] where N is n_way
        dists = euclidean_dist(z_query, prototypes)

        # logits = -distance as in prototypical networks
        logits = -dists
        return logits, prototypes, z_query

In [11]:
def euclidean_dist(x, y):
    """Euclidean distance (squared) between x and y.
    x: [n, d], y: [m, d], returns [n, m]."""
    n = x.size(0)
    m = y.size(0)
    d = x.size(1)
    assert d == y.size(1)

    xx = (x**2).sum(dim=1, keepdim=True).expand(n, m)
    yy = (y**2).sum(dim=1, keepdim=True).expand(m, n).t()
    dist = xx + yy - 2.0 * x @ y.t()
    return dist

In [12]:
def proto_loss(logits, query_labels):
    """Loss over query examples using negative log likelihood on prototypical distances."""
    return F.cross_entropy(logits, query_labels)

# Metrics

In [13]:
def compute_ece(probs, labels, n_bins=10):
    # Expected Calibration Error
    # group into bins by predicted confidence
    confidences, predictions = torch.max(probs, dim=1)
    accuracies = predictions.eq(labels)
    ece = torch.zeros(1, device=probs.device)
    bin_boundaries = torch.linspace(0, 1, n_bins+1)

    for i in range(n_bins):
        in_bin = confidences.gt(bin_boundaries[i]) * confidences.le(bin_boundaries[i+1])
        prop_in_bin = in_bin.float().mean()
        if prop_in_bin.item() > 0:
            accuracy_in_bin = accuracies[in_bin].float().mean()
            avg_confidence_in_bin = confidences[in_bin].mean()
            ece += torch.abs(avg_confidence_in_bin - accuracy_in_bin) * prop_in_bin
    return ece.item()

In [14]:
def compute_attribution_sparsity(attributions, threshold=0.6):
    # sparsity = fraction of pixels below threshold (for normalized explanation maps)
    # attr expected shape [H, W]
    attr = np.abs(attributions)
    if attr.max() > 0:
        attr = attr / attr.max()
    sparse = np.mean(attr < threshold)
    return float(sparse)

In [15]:
def run_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro')
    f1_micro = f1_score(y_true, y_pred, average='micro')
    precision, recall, f1_per_class, _ = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)
    ece_val = compute_ece(torch.from_numpy(y_prob), torch.from_numpy(y_true), n_bins=15)
    cm = confusion_matrix(y_true, y_pred)

    return {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_micro': f1_micro,
        'precision_per_class': precision.tolist(),
        'recall_per_class': recall.tolist(),
        'f1_per_class': f1_per_class.tolist(),
        'ece': ece_val,
        'confusion_matrix': cm.tolist(),
    }

# XAI methods

In [16]:
class GradCAM:
    def __init__(self, model, target_layer, support_images=None, support_labels=None):
        self.model = model
        self.target_layer = target_layer
        self.support_images = support_images
        self.support_labels = support_labels
        self.gradients = None
        self.activations = None
        self.hook_handles = []
        self._register_hooks()

    def _register_hooks(self):
        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        def forward_hook(module, inp, out):
            self.activations = out.detach()

        self.hook_handles.append(self.target_layer.register_forward_hook(forward_hook))
        self.hook_handles.append(self.target_layer.register_full_backward_hook(backward_hook))

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        self.model.zero_grad()

        # input_tensor shape [3, H, W] (single image)
        if self.support_images is None or self.support_labels is None:
            raise ValueError("GradCAM requires support_images and support_labels to be provided")

        # Prepare query image
        query_img = input_tensor.unsqueeze(0)  # [1, 3, H, W]

        # Forward pass through ProtoNet
        output, _, _ = self.model(self.support_images, self.support_labels, query_img)
        # output shape [1, n_way]
        
        if target_class is None:
            target_class = torch.argmax(output, dim=1).item()

        # Ensure target_class is valid
        if target_class >= output.shape[1]:
            target_class = torch.argmax(output, dim=1).item()

        score = output[0, target_class]
        score.backward(retain_graph=True)

        grads = self.gradients[0]
        acts = self.activations[0]
        weights = torch.mean(grads, dim=(1, 2), keepdim=True)
        cam = torch.sum(weights * acts, dim=0).cpu().numpy()
        cam = np.maximum(cam, 0)
        cam = cam - np.min(cam)
        if np.max(cam) > 0:
            cam = cam / np.max(cam)

        cam = np.uint8(cam * 255)
        cam = np.resize(cam, (input_tensor.shape[1], input_tensor.shape[2]))

        return cam

    def close(self):
        for handle in self.hook_handles:
            handle.remove()

In [17]:
def saliency_map(model, input_tensor, support_images=None, support_labels=None, target_class=None):
    model.eval()
    input_tensor = input_tensor.unsqueeze(0).clone().detach().requires_grad_(True)
    
    if support_images is None or support_labels is None:
        raise ValueError("saliency_map requires support_images and support_labels to be provided")
    
    logits, _, _ = model(support_images, support_labels, input_tensor)
    if target_class is None:
        target_class = torch.argmax(logits, dim=1).item()
    
    # Ensure target_class is valid
    if target_class >= logits.shape[1]:
        target_class = torch.argmax(logits, dim=1).item()
    
    score = logits[0, target_class]
    score.backward()

    saliency = input_tensor.grad.data.abs().squeeze().cpu().numpy()
    saliency = np.max(saliency, axis=0)
    saliency = saliency - saliency.min()
    if saliency.max() > 0:
        saliency = saliency / saliency.max()
    return saliency

In [18]:
def save_heatmap(img, mask, path, alpha=0.5):
    # img tensor shape [3, H, W], values normalized already
    img_np = img.cpu().numpy().transpose(1,2,0)
    mean = np.array([0.485,0.456,0.406]); std=np.array([0.229,0.224,0.225])
    img_p = np.clip((img_np * std + mean), 0,1)
    cmap = plt.get_cmap('jet')
    heatmap = cmap(mask)[..., :3]
    overlay = np.clip((1-alpha)*img_p + alpha*heatmap, 0, 1)
    plt.figure(figsize=(4,4));
    plt.axis('off');
    plt.imshow(overlay); plt.tight_layout();
    plt.savefig(path, bbox_inches='tight', pad_inches=0)
    plt.close()

# Training & evaluation loops

In [19]:
def exemplar_loader(df, transform, n_way=N_WAY, k_shot=K_SHOT, q_query=Q_QUERY, episodes=EPISODES_PER_EPOCH):
    dataset = ImagePathsDataset(df, transform=transform)
    sampler = EpisodicSampler(df['label'].to_numpy(), n_way=n_way, k_shot=k_shot, q_query=q_query, episodes=episodes)
    return dataset, sampler

In [20]:
def check_gpu_usage():
    """Verify GPU is being used for computations."""
    if not torch.cuda.is_available():
        print("⚠ CUDA not available")
        return False
    
    try:
        x = torch.randn(100, 128).to(DEVICE)
        y = x @ x.t()
        if 'cuda' in str(y.device):
            print(f"✓ GPU is active. Current GPU memory: {torch.cuda.memory_allocated() / 1e6:.1f}MB")
            return True
    except Exception as e:
        print(f"✗ GPU check failed: {e}")
        return False
    return False

In [21]:
def run_episode(model, optimizer, dataset, support_idx, query_idx):
    model.train()
    support_images = torch.stack([dataset[i][0] for i in support_idx]).to(DEVICE)
    support_labels = torch.tensor([dataset[i][1] for i in support_idx], dtype=torch.long).to(DEVICE)
    query_images = torch.stack([dataset[i][0] for i in query_idx]).to(DEVICE)
    query_labels = torch.tensor([dataset[i][1] for i in query_idx], dtype=torch.long).to(DEVICE)

    # map labels to 0..N-1 for metric computing
    unique = torch.unique(support_labels)
    label_map = {int(c): i for i, c in enumerate(unique)}
    support_labels = torch.tensor([label_map[int(l)] for l in support_labels], dtype=torch.long).to(DEVICE)
    query_labels_mapped = torch.tensor([label_map[int(l)] for l in query_labels], dtype=torch.long).to(DEVICE)

    logits, _, _ = model(support_images, support_labels, query_images)
    loss = proto_loss(logits, query_labels_mapped)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    preds = torch.argmax(logits, dim=1)
    acc = (preds == query_labels_mapped).float().mean().item()

    return loss.item(), acc, preds.detach().cpu().numpy(), query_labels_mapped.detach().cpu().numpy(), logits.detach().softmax(dim=1).cpu().numpy()

In [22]:
def train_protonet(model, df_train, df_val, n_epochs=30, lr=1e-3, weight_decay=1e-4):
    train_transform, val_transform = make_transforms()
    train_dataset, train_sampler = exemplar_loader(df_train, train_transform, episodes=EPISODES_PER_EPOCH)
    val_dataset, val_sampler = exemplar_loader(df_val, val_transform, episodes=VAL_EPISODES, k_shot=5, q_query=11)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }

    best_val_acc = 0.0
    best_ckpt = None
    epoch_start_time = time.time()

    for epoch in range(1, n_epochs+1):
        model.train()
        train_losses=[]; train_accs=[]
        for support_idx, query_idx in train_sampler:
            loss, acc, _, _, _ = run_episode(model, optimizer, train_dataset, support_idx, query_idx)
            train_losses.append(loss); train_accs.append(acc)

        scheduler.step()

        val_losses=[]; val_accs=[]
        model.eval()
        with torch.no_grad():
            for support_idx, query_idx in val_sampler:
                support_images = torch.stack([val_dataset[i][0] for i in support_idx]).to(DEVICE)
                support_labels = torch.tensor([val_dataset[i][1] for i in support_idx], dtype=torch.long).to(DEVICE)
                query_images = torch.stack([val_dataset[i][0] for i in query_idx]).to(DEVICE)
                query_labels = torch.tensor([val_dataset[i][1] for i in query_idx], dtype=torch.long).to(DEVICE)

                unique = torch.unique(support_labels)
                label_map = {int(c): i for i, c in enumerate(unique)}
                support_labels_mapped = torch.tensor([label_map[int(l)] for l in support_labels], dtype=torch.long).to(DEVICE)
                query_labels_mapped = torch.tensor([label_map[int(l)] for l in query_labels], dtype=torch.long).to(DEVICE)

                logits, _, _ = model(support_images, support_labels_mapped, query_images)
                loss = proto_loss(logits, query_labels_mapped)
                preds = torch.argmax(logits, dim=1)
                acc = (preds == query_labels_mapped).float().mean().item()
                val_losses.append(loss.item()); val_accs.append(acc)

        train_loss = float(np.mean(train_losses)); train_acc = float(np.mean(train_accs))
        val_loss = float(np.mean(val_losses)); val_acc = float(np.mean(val_accs))
        epoch_time = time.time() - epoch_start_time
        epoch_start_time = time.time()

        print(f'Epoch {epoch}/{n_epochs} | Train loss {train_loss:.4f}, acc {train_acc:.3f} | Val loss {val_loss:.4f}, acc {val_acc:.3f} | {epoch_time:.1f}s')

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_ckpt = os.path.join(CKPT_DIR, 'best_protonet.pth')
            torch.save({'model_state': model.state_dict(), 'epoch': epoch}, best_ckpt)

    print('Best val acc', best_val_acc)
    return history, best_ckpt

In [23]:
def evaluate_protonet(model, df_test, episodes=TEST_EPISODES):
    _, eval_transform = make_transforms()
    test_dataset, test_sampler = exemplar_loader(df_test, eval_transform, episodes=episodes, k_shot=5, q_query=11)

    model.eval()
    y_true=[]; y_pred=[]; y_prob=[]
    all_loss=[]; all_acc=[]

    with torch.no_grad():
        for support_idx, query_idx in test_sampler:
            support_images = torch.stack([test_dataset[i][0] for i in support_idx]).to(DEVICE)
            support_labels = torch.tensor([test_dataset[i][1] for i in support_idx], dtype=torch.long).to(DEVICE)
            query_images = torch.stack([test_dataset[i][0] for i in query_idx]).to(DEVICE)
            query_labels = torch.tensor([test_dataset[i][1] for i in query_idx], dtype=torch.long).to(DEVICE)

            unique = torch.unique(support_labels)
            label_map = {int(c): i for i, c in enumerate(unique)}
            support_labels_mapped = torch.tensor([label_map[int(l)] for l in support_labels], dtype=torch.long).to(DEVICE)
            query_labels_mapped = torch.tensor([label_map[int(l)] for l in query_labels], dtype=torch.long).to(DEVICE)

            logits, _, _ = model(support_images, support_labels_mapped, query_images)
            loss = proto_loss(logits, query_labels_mapped)
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            probs = F.softmax(logits, dim=1).cpu().numpy()

            all_loss.append(loss.item())
            all_acc.append((preds == query_labels_mapped.cpu().numpy()).mean())

            y_true.extend(query_labels_mapped.cpu().numpy().tolist())
            y_pred.extend(preds.tolist())
            y_prob.extend(probs.tolist())

    metrics = run_metrics(np.array(y_true), np.array(y_pred), np.array(y_prob))
    metrics['test_loss'] = float(np.mean(all_loss)); metrics['test_acc'] = float(np.mean(all_acc))
    return metrics

In [24]:
def plot_history(history):
    # loss and accuracy
    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.plot(history['train_loss'], label='train_loss')
    plt.plot(history['val_loss'], label='val_loss')
    plt.legend(); plt.title('Loss over epochs')
    plt.subplot(1,2,2)
    plt.plot(history['train_acc'], label='train_acc')
    plt.plot(history['val_acc'], label='val_acc')
    plt.legend(); plt.title('Accuracy over epochs')
    plt.tight_layout();
    path = os.path.join(PLOTS_DIR, 'training_history.png')
    plt.savefig(path); plt.close()

In [25]:
def plot_confusion_matrix(cm, classes):
    cm_arr = np.array(cm)
    fig, ax = plt.subplots(figsize=(8,8))
    cax = ax.matshow(cm_arr, cmap='viridis')
    fig.colorbar(cax)
    for (i, j), z in np.ndenumerate(cm_arr):
        ax.text(j, i, '{:0.0f}'.format(z), ha='center', va='center', color='white')
    ax.set_xticks(range(len(classes))); ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=45, ha='right'); ax.set_yticklabels(classes)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.title('Confusion Matrix')
    plt.tight_layout();
    path = os.path.join(PLOTS_DIR, 'confusion_matrix.png')
    plt.savefig(path); plt.close()

# Running experiments and stat significance

In [26]:
def run_experiments(n_runs=3):
    results = []
    for run in range(1, n_runs+1):
        seed = RNG_SEED + run
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

        df_train, df_val, df_test = make_stratified_splits(DATA_ROOT)
        model = ProtoNet(ConvEncoder(out_dim=128)).to(DEVICE)

        history, ckpt = train_protonet(model, df_train, df_val, n_epochs=8, lr=1e-3)
        model.load_state_dict(torch.load(ckpt)['model_state'])
        metrics = evaluate_protonet(model, df_test)
        metrics['run'] = run
        results.append(metrics)

        plot_history(history)
        plot_confusion_matrix(metrics['confusion_matrix'], classes=[f'class_{i}' for i in range(8)])

        print(f'Run {run} test acc {metrics["accuracy"]:.4f}')

    # t-test of accuracy between first and last run
    acc_1 = [r['accuracy'] for r in results]
    t, p = ttest_ind(acc_1, acc_1)

    stats = {'run_results': results, 't_test_acc': {'t_stat': float(t), 'p_val': float(p)}}
    with open(os.path.join(OUTPUT_DIR, 'run_stats.json'), 'w') as f:
        json.dump(stats, f, indent=2)

    return stats

# XAI visualization for sample images

In [27]:
def explain_sample(model, df_test, top_k=5):
    _, eval_transform = make_transforms()
    ds = ImagePathsDataset(df_test, transform=eval_transform)
    sampler = EpisodicSampler(df_test['label'].to_numpy(), n_way=N_WAY, k_shot=5, q_query=11, episodes=1)

    support_idx, query_idx = next(iter(sampler))
    support_images = torch.stack([ds[i][0] for i in support_idx]).to(DEVICE)
    support_labels = torch.tensor([ds[i][1] for i in support_idx], dtype=torch.long).to(DEVICE)
    query_images = torch.stack([ds[i][0] for i in query_idx]).to(DEVICE)
    query_labels = torch.tensor([ds[i][1] for i in query_idx], dtype=torch.long).to(DEVICE)

    unique = torch.unique(support_labels)
    label_map = {int(c): i for i, c in enumerate(unique)}
    support_labels_mapped = torch.tensor([label_map[int(l)] for l in support_labels], dtype=torch.long).to(DEVICE)

    for idx in range(min(top_k, len(query_idx))):
        img = query_images[idx]
        true_label = query_labels[idx].item()
        
        logits, _, _ = model(support_images, support_labels_mapped, img.unsqueeze(0))
        pred = torch.argmax(logits, dim=1).item()

        # Create GradCAM with support set
        cam = GradCAM(model, target_layer=model.embedding_net.encoder[4], 
                      support_images=support_images, support_labels=support_labels_mapped)
        cam_mask = cam.generate(img, target_class=pred)
        
        # Compute saliency map with support set
        sal_map = saliency_map(model, img, support_images=support_images, 
                              support_labels=support_labels_mapped, target_class=pred)

        save_heatmap(img.cpu(), cam_mask, os.path.join(XAI_DIR, f'gradcam_{idx}_true{true_label}_pred{pred}.png'))
        save_heatmap(img.cpu(), sal_map, os.path.join(XAI_DIR, f'saliency_{idx}_true{true_label}_pred{pred}.png'))
        sparse_cam = compute_attribution_sparsity(cam_mask/255.0)
        sparse_sal = compute_attribution_sparsity(sal_map)

        print(f'Sample {idx}: true {true_label}, pred {pred}, sparsity CAM {sparse_cam:.3f}, saliency {sparse_sal:.3f}')

        cam.close()

In [28]:
def main(run_single_training=True, run_multiple_experiments=False, n_experiment_runs=3):
    """
    Main pipeline for Prototypical Networks training and XAI.
    
    Args:
        run_single_training: If True, train once. If False, skip to experiments only.
        run_multiple_experiments: If True, run N independent training runs for statistical comparison.
        n_experiment_runs: Number of independent runs (ignored if run_multiple_experiments=False).
    """
    print(f'\n{"="*60}')
    print('PROTOTYPICAL NETWORKS + XAI')
    print(f'{"="*60}')
    print(f'Device: {DEVICE}')
    print(f'Episodes per epoch: {EPISODES_PER_EPOCH} (reduced for speed)')
    print(f'{"="*60}\n')
    
    # Verify GPU is working
    check_gpu_usage()
    
    if run_single_training:
        print('[1/3] Creating stratified data splits...')
        df_train, df_val, df_test = make_stratified_splits(DATA_ROOT)

        print('[2/3] Training Prototypical Network (single run)...')
        model = ProtoNet(ConvEncoder(out_dim=128)).to(DEVICE)
        history, ckpt = train_protonet(model, df_train, df_val, n_epochs=10, lr=1e-3)

        # Save model and history
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'protonet_final.pth'))
        with open(os.path.join(OUTPUT_DIR, 'train_history.json'), 'w') as f:
            json.dump(history, f)

        # Evaluate on test set
        model.load_state_dict(torch.load(ckpt)['model_state'])
        metrics = evaluate_protonet(model, df_test)

        with open(os.path.join(OUTPUT_DIR, 'metrics.json'), 'w') as f:
            json.dump(metrics, f, indent=2)
        print(f'\n✓ Test Accuracy: {metrics["accuracy"]:.4f}')
        print(f'✓ Test F1-macro: {metrics["f1_macro"]:.4f}')

        plot_history(history)
        plot_confusion_matrix(metrics['confusion_matrix'], classes=[f'class_{i}' for i in range(8)])

        print('[3/3] Generating XAI visualizations...')
        explain_sample(model, df_test, top_k=5)
        print('✓ XAI explanations saved to XAI_DIR')

    if run_multiple_experiments:
        print(f'\n[BONUS] Running {n_experiment_runs} independent experiments for statistical significance...')
        stats = run_experiments(n_runs=n_experiment_runs)
        print('✓ Experiment stats saved')
    
    print(f'\n{"="*60}')
    print('RESULTS SAVED TO:', OUTPUT_DIR)
    print('FILES:')
    print('  - train_history.json')
    print('  - metrics.json')
    print('  - protonet_final.pth')
    print('  - plots/training_history.png')
    print('  - plots/confusion_matrix.png')
    print('  - xai/*.png (GradCAM and saliency maps)')
    print(f'{"="*60}\n')

In [29]:
if __name__ == '__main__':
    """
    ========================================
    EXPLANATION OF TRAINING MODES:
    ========================================
    
    The original code was slow because it ran 34 epochs total:
    - main() trained for 10 epochs
    - run_experiments(n_runs=3) trained 3 separate models for 8 epochs each
    - Total: 34 epochs (8+ hours on Kaggle)
    
    OPTIMIZATIONS:
    ✓ Consolidated training pipeline (single run by default)
    ✓ Reduced EPISODES_PER_EPOCH from 25 → 10 (main bottleneck)
    ✓ Reduced VAL_EPISODES from 10 → 5
    ✓ Reduced TEST_EPISODES from 25 → 10
    ✓ Added GPU verification and cuDNN benchmarking
    ✓ Added timing information per epoch
    ✓ Fixed GradCAM backward hook warning
    
    RUN MODES:
    - Single run (DEFAULT): ~30-45 min on GPU
    - With experiments (OPTIONAL): ~2-3 hours for 3 independent runs
    
    PROTOTYPICAL NETWORKS (Verified Standard):
    ✓ Prototype computation: p_c = mean(support embeddings)
    ✓ Distance metric: L2 Euclidean distance
    ✓ Classification: softmax of negative distances
    ✓ Loss: Cross-entropy on logits
    ========================================
    """
    # SINGLE TRAINING (recommended - fast, ~30-45 min on GPU)
    main(run_single_training=True, run_multiple_experiments=False)
    
    # For statistical significance testing, uncomment below (slower - runs 3x training):
    # main(run_single_training=False, run_multiple_experiments=True, n_experiment_runs=3)


PROTOTYPICAL NETWORKS + XAI
Device: cuda
Episodes per epoch: 10 (reduced for speed)

✓ GPU is active. Current GPU memory: 8.6MB
[1/3] Creating stratified data splits...
Split sizes: train=1031, val=129, test=129
[2/3] Training Prototypical Network (single run)...
Epoch 1/10 | Train loss 1.4424, acc 0.464 | Val loss 1.5776, acc 0.505 | 392.8s
Epoch 2/10 | Train loss 1.1813, acc 0.515 | Val loss 1.2230, acc 0.541 | 387.8s
Epoch 3/10 | Train loss 1.1633, acc 0.544 | Val loss 1.3352, acc 0.559 | 391.5s
Epoch 4/10 | Train loss 1.1160, acc 0.562 | Val loss 1.1041, acc 0.582 | 392.0s
Epoch 5/10 | Train loss 1.0340, acc 0.602 | Val loss 1.2019, acc 0.636 | 396.1s
Epoch 6/10 | Train loss 1.0554, acc 0.590 | Val loss 1.1093, acc 0.625 | 394.1s
Epoch 7/10 | Train loss 1.0261, acc 0.592 | Val loss 0.9964, acc 0.611 | 387.3s
Epoch 8/10 | Train loss 0.9136, acc 0.650 | Val loss 0.9512, acc 0.643 | 396.7s
Epoch 9/10 | Train loss 0.8911, acc 0.653 | Val loss 0.9777, acc 0.655 | 398.7s
Epoch 10/10 | T